In [ ]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [ ]:
# data loading
train, test = load_data()

In [ ]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거
X_features['var3'] = X_features['var3'].replace(-999999, 2)
# var3 의 최소값 -99999 를 최빈값으로 변경하기

In [ ]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features,
  y_labels,
)

In [ ]:
# 2) Scaler 생성 (train에만 fit)
scaler = StandardScaler()
scaler.fit(X_train)

스케일링 후 데이터프레임:


,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,...,saldo_medio_var29_ult3,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38
0,-0.075835,-0.788249,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.427183
1,-0.075835,0.060753,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.372038
2,-0.075835,-0.788249,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.273191
3,-0.075835,0.292298,-0.053388,0.361427,0.138158,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.291398
4,-0.075835,0.446662,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,0.000412


In [ ]:
# 3) train, val, test에 동일한 scaler 적용
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

In [ ]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # 재학습 방지 — 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [17]:
# lgbm , logisticregression
lgbm_clf = LGBMClassifier(
    random_state      = 0,
    n_estimators      = 100,
    num_leaves        = 31,
    min_child_samples = 10,
)

lgbm_clf.fit(X_train, y_train)

pred = lgbm_clf.predict(X_val)
pred_proba = lgbm_clf.predict_proba(X_val)[:,1]

get_model_train_eval(
    lgbm_clf,
    'LightGBM_100_num31_min10',
    X_train, X_val,
    y_train, y_val
)

best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (pred_proba >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# # 최적 threshold로 성능 출력
# pred_best = (pred_proba >= best_threshold).astype(int)
# get_clf_eval(y_val, pred_best, pred_proba)
threshold_model = ThresholdModel(lgbm_clf, best_threshold)

get_model_train_eval(
    threshold_model,
    "LightGBM_100_num31_min10_thr",
    X_train, X_val,
    y_train, y_val
)

[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023295 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14600
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 263
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039562 -> initscore=-3.189521
[LightGBM] [Info] Start training from score -3.189521
[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023347 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14600
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 263
[LightGBM] [Info